# Part 3 Starter Notebook

This notebook loads an AIME 2024 dataset, runs a model on each problem, extracts an AIME-style final answer, and grades the outputs.

In [ ]:
import re

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B"
# or allenai/Olmo-3-7B-Thinking
DATASET_NAME = "OpenRLHF/aime-2024"
MAX_NEW_TOKENS = 38912

## Loading the model and the data

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset(DATASET_NAME, split="train")


## Evaluation helpers

In [ ]:
def strip_thinking_trace(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|begin_of_thought\|>.*?<\|end_of_thought\|>", "", text, flags=re.DOTALL)
    return text.strip()



def extract_answer(text: str, mode="exact_match") -> int | None:
    """Extract an AIME-style integer answer from a model completion."""
    answer_text = strip_thinking_trace(text)
    if not answer_text:
        if mode == "exact_match":
            return None
        else:
            answer_text = text  # fall back to full text


    # 1. Boxed LaTeX answer: \boxed{123}
    if mode == "exact_match":
        boxed = re.findall(r"\\boxed\{(\d+)\}", answer_text)
        if boxed:
            val = int(boxed[-1])
            return val
        else:
            return None

    elif mode == "flexible_extract":
        # 2. "The answer is N" or "answer: N" patterns
        patterns = [
            r"(?:the\s+)?answer\s+is\s+[:\s]*(\d+)",
            r"answer[:\s]+(\d+)",
            r"=\s*(\d+)\s*$",
            r"(?:therefore|thus|so),?\s+(\d+)\s*(?:\.|$)",
        ]
        for pattern in patterns:
            matches = re.findall(pattern, answer_text, re.IGNORECASE)
            if matches:
                val = int(matches[-1])
                return val

        # 3. Last integer in [0, 999] in the answer portion
        integers = re.findall(r"\b(\d{1,3})\b", answer_text)
        for candidate in reversed(integers):
            val = int(candidate)
            return val
        return None

def get_thinking_length(txt):
  """Determine the number of thinking tokens generated in a model's output"""
  match = re.search(r"<think>(.*?)</think>", txt, flags=re.DOTALL)
  if match:
    trace = match.group(1).strip()
    thinking_tokens = tokenizer.encode(trace, add_special_tokens=False)
    return len(thinking_tokens)
  else:
    return MAX_NEW_TOKENS

## Inference

In [ ]:
import matplotlib.pyplot as plt

def run_eval(enable_thinking=True):
  model.to(device)
  model.eval()

  ex_records = []
  thinking_lengths = []
  fl_records = []

  for i, example in enumerate(dataset):
      print(f"Enable Thinking: {enable_thinking}, Progress: {i+1} out of {len(dataset)}")

      problem = example["prompt"]
      gold_answer = int(example["label"])

      messages = [{"role": "system", "content": "You are a careful competition math assistant.  Always output your final answer in \\boxed{}."},
                  {"role": "user", "content": problem[0].get('content')}]
      prompt = tokenizer.apply_chat_template(
          messages,
          tokenize=False,
          add_generation_prompt=True,
          enable_thinking=enable_thinking,
      )

      inputs = tokenizer(prompt, return_tensors="pt").to(device)

      with torch.no_grad():
          output_ids = model.generate(
              **inputs,
              max_new_tokens=MAX_NEW_TOKENS,
              do_sample=False,
              temperature=None,
              top_p=None,
          )

      # Decode only the newly generated tokens
      generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
      model_output = tokenizer.decode(generated_ids, skip_special_tokens=True)

      thinking_lengths.append(get_thinking_length(model_output))

      ex_extracted = extract_answer(model_output, mode='exact_match')
      if ex_extracted is not None:
          ex_correct = ex_extracted == gold_answer
      else:
          ex_correct = False

      ex_records.append({
          "problem": problem,
          "gold_answer": gold_answer,
          "model_output": model_output,
          "extracted_answer": ex_extracted,
          "correct": ex_correct,
      })

      fl_extracted = extract_answer(model_output, mode='flexible_extract')
      if fl_extracted is not None:
          fl_correct = fl_extracted == gold_answer
      else:
          fl_correct = False

      fl_records.append({
          "problem": problem,
          "gold_answer": gold_answer,
          "model_output": model_output,
          "extracted_answer": fl_extracted,
          "correct": fl_correct,
      })

      print(f"[{i+1}/{len(dataset)}] EXACT_MATCH : gold={gold_answer} pred={ex_extracted} correct={ex_correct}")
      print(f"[{i+1}/{len(dataset)}] FLEX_EXTRACT: gold={gold_answer} pred={fl_extracted} correct={fl_correct}")
      print(f"[{i+1}/{len(dataset)}] thinking length={thinking_lengths[-1]}\n")

  ex_results_df = pd.DataFrame(ex_records)
  ex_accuracy = ex_results_df["correct"].mean()
  fl_results_df = pd.DataFrame(fl_records)
  fl_accuracy = fl_results_df["correct"].mean()

  return ex_results_df, ex_accuracy, fl_results_df, fl_accuracy, thinking_lengths

In [ ]:
# Run with thinking
print("=" * 30)
print("Evaluation: Thinking")
print("=" * 30)
ex_results_with_thinking, ex_acc_with, fl_results_with_thinking, fl_acc_with, thinking_lengths = run_eval(
    enable_thinking=True
)
print(f"\nAccuracy with thinking (exact_match): {ex_acc_with:.4f}")
print(f"\nAccuracy with thinking (flexible_extract): {fl_acc_with:.4f}")

# Plot histogram of thinking lengths
if thinking_lengths:
    plt.figure(figsize=(10, 6))
    plt.hist(thinking_lengths, bins=20, edgecolor='black', alpha=0.7)
    plt.xlabel('Number of Tokens in Thinking Trace')
    plt.ylabel('Frequency')
    plt.title(f'Distribution of Thinking Trace Lengths\nMean: {sum(thinking_lengths)/len(thinking_lengths):.1f}, Median: {sorted(thinking_lengths)[len(thinking_lengths)//2]}')
    plt.grid(alpha=0.3)
    plt.savefig('thinking_lengths_histogram.png')
    plt.show()

    print(f"\nThinking trace statistics:")
    print(f"  Mean length: {sum(thinking_lengths)/len(thinking_lengths):.1f} tokens")
    print(f"  Median length: {sorted(thinking_lengths)[len(thinking_lengths)//2]} tokens")
    print(f"  Min length: {min(thinking_lengths)} tokens")
    print(f"  Max length: {max(thinking_lengths)} tokens")

In [ ]:
# Run without thinking
print("\n" + "=" * 30)
print("Evaluation: No Thinking")
print("=" * 30)
ex_results_without_thinking, ex_acc_without, fl_results_without, fl_acc_without, _ = run_eval(
    enable_thinking=False
)
print(f"\nAccuracy without thinking (exact_match): {ex_acc_without:.4f}")
print(f"\nAccuracy without thinking (flexible_extract): {fl_acc_without:.4f}")

# 2.2 Scaling Experiments

## Parallel Scaling

In [ ]:
def run_parallel_eval(enable_thinking=True):
  model.to(device)
  model.eval()

  tokens_per_sample = 4000

  m_lengths = [1, 2, 4, 6, 8]

  for i, example in enumerate(dataset):
      thinking_length = 0

      ex_majority_vote = {}
      ex_best_of_m = {}
      fl_majority_vote = {}
      fl_best_of_m = {}
      ex_answers = []
      fl_answers = []

      print(f"Enable Thinking: {enable_thinking}, Progress: {i+1} out of {len(dataset)}")

      problem = example["prompt"]
      gold_answer = int(example["label"])

      messages = [{"role": "system", "content": "You are a careful competition math assistant.  Always output your final answer in \\boxed{}."},
                  {"role": "user", "content": problem[0].get('content')}]
      prompt = tokenizer.apply_chat_template(
          messages,
          tokenize=False,
          add_generation_prompt=True,
          enable_thinking=enable_thinking,
      )

      inputs = tokenizer(prompt, return_tensors="pt").to(device)

      for j in range(m_lengths[-1]):
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=tokens_per_sample,
                do_sample=True,
                temperature=0.6,
                top_p=0.95,
                top_k=50,
            )

        # check if the generation has already finished (151645 is <|im_end|>)
        if 151645 not in output_ids[0][inputs.input_ids.size(-1):].tolist():
            # check if the thinking process has finished (151668 is </think>)
            # and prepare the second model input
            if 151668 not in output_ids[0][inputs.input_ids.size(-1):].tolist():
                # print("thinking budget is reached")
                early_stopping_text = "\n\nConsidering the limited time by the user, I have to stop thinking and give my final answer in \\boxed{}.\n</think>\n\n Final Answer: \\boxed{"
                early_stopping_ids = tokenizer([early_stopping_text], return_tensors="pt", return_attention_mask=False).input_ids.to(model.device)
                input_ids = torch.cat([output_ids, early_stopping_ids], dim=-1)

                attention_mask = torch.ones_like(input_ids, dtype=torch.int64)

                # second generation
                output_ids = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=1,
                    do_sample=False,
                )

        # Decode only the newly generated tokens
        generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        model_output = tokenizer.decode(generated_ids, skip_special_tokens=True)

        ex_extracted = extract_answer(model_output, mode='exact_match')

        fl_extracted = extract_answer(model_output, mode='flexible_extract')

        if ex_extracted is not None:
            ex_correct = ex_extracted == gold_answer
            ex_answers.append(ex_extracted)
        else:
            ex_correct = False
            ex_answers.append(gold_answer - 1000)

        if fl_extracted is not None:
            fl_correct = fl_extracted == gold_answer
            fl_answers.append(fl_extracted)
        else:
            fl_correct = False
            fl_answers.append(gold_answer - 1000)

        print(f"\n\n[{i+1}/{len(dataset)}]({j+1}) EXACT_MATCH : gold={gold_answer} pred={ex_extracted} correct={ex_correct}")
        print(f"[{i+1}/{len(dataset)}]({j+1}) FLEX_EXTRACT: gold={gold_answer} pred={fl_extracted} correct={fl_correct}")
        print(f"[{i+1}/{len(dataset)}]({j+1}) thinking length={get_thinking_length(model_output)}")

        thinking_length += get_thinking_length(model_output)

        # most frequent answer
        ex_majority_vote[j] = max(set(ex_answers), key=ex_answers.count)
        fl_majority_vote[j] = max(set(fl_answers), key=fl_answers.count)

        # answer closest to true answer
        ex_best_of_m[j] = min(ex_answers, key=lambda x: abs(x - gold_answer))
        fl_best_of_m[j] = min(fl_answers, key=lambda x: abs(x - gold_answer))

        # determine majority vote and best-of-m picks for current token budget
        ex_maj_acc = sum([ex_majority_vote[k] == gold_answer for k in range(j+1)]) / (j+1)
        ex_bon_acc = sum([ex_best_of_m[k] == gold_answer for k in range(j+1)]) / (j+1)

        fl_maj_acc = sum([fl_majority_vote[k] == gold_answer for k in range(j+1)]) / (j+1)
        fl_bon_acc = sum([fl_best_of_m[k] == gold_answer for k in range(j+1)]) / (j+1)

        print(f"m: {j+1} Exact Majority Acc: {ex_maj_acc} Exact Best-of-M Acc: {ex_bon_acc}")
        print(f"m: {j+1} Flex Majority Acc: {fl_maj_acc} Flex Best-of-M Acc: {fl_bon_acc}")
        print(f"Total Number of tokens used: {thinking_length}")

  return 

In [ ]:
# Run parallel-scaling with thinking
print("=" * 30)
print("Evaluation: Parallel Scaling")
print("=" * 30)
run_parallel_eval(enable_thinking=True)

# 2.4 Parallel Optimizations

## Variable Temperatures

In [ ]:
def run_temperature_eval(enable_thinking=True):
  model.to(device)
  model.eval()

  tokens_per_sample = 4000

  m_lengths = [1, 2, 4, 6, 8]
  temperatures = [0.35, 0.6, 0.85]

  for i, example in enumerate(dataset):
      thinking_length = 0

      ex_majority_vote = {}
      ex_best_of_m = {}
      fl_majority_vote = {}
      fl_best_of_m = {}
      ex_answers = []
      fl_answers = []

      problem = example["prompt"]
      gold_answer = int(example["label"])

      messages = [{"role": "system", "content": "You are a careful competition math assistant.  Always output your final answer in \\boxed{}."},
                  {"role": "user", "content": problem[0].get('content')}]
      prompt = tokenizer.apply_chat_template(
          messages,
          tokenize=False,
          add_generation_prompt=True,
          enable_thinking=enable_thinking,
      )

      inputs = tokenizer(prompt, return_tensors="pt").to(device)

      for temperature in temperatures:
        print(f"Temperature: {temperature}, Progress: {i+1} out of {len(dataset)}")
        for j in range(m_lengths[-1]):
          with torch.no_grad():
              output_ids = model.generate(
                  **inputs,
                  max_new_tokens=tokens_per_sample,
                  do_sample=True,
                  temperature=temperature,  # same approach as before, but varying the temperature parameter for the model
                  top_p=0.95,
                  top_k=50,
              )

          # check if the generation has already finished (151645 is <|im_end|>)
          if 151645 not in output_ids[0][inputs.input_ids.size(-1):].tolist():
              # check if the thinking process has finished (151668 is </think>)
              # and prepare the second model input
              if 151668 not in output_ids[0][inputs.input_ids.size(-1):].tolist():
                  # print("thinking budget is reached")
                  early_stopping_text = "\n\nConsidering the limited time by the user, I have to stop thinking and give my final answer in \\boxed{}.\n</think>\n\n Final Answer: \\boxed{"
                  early_stopping_ids = tokenizer([early_stopping_text], return_tensors="pt", return_attention_mask=False).input_ids.to(model.device)
                  input_ids = torch.cat([output_ids, early_stopping_ids], dim=-1)

                  attention_mask = torch.ones_like(input_ids, dtype=torch.int64)

                  # second generation
                  output_ids = model.generate(
                      input_ids=input_ids,
                      attention_mask=attention_mask,
                      max_new_tokens=1,
                      do_sample=False,
                  )

          # Decode only the newly generated tokens
          generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
          model_output = tokenizer.decode(generated_ids, skip_special_tokens=True)

          ex_extracted = extract_answer(model_output, mode='exact_match')

          fl_extracted = extract_answer(model_output, mode='flexible_extract')

          if ex_extracted is not None:
              ex_correct = ex_extracted == gold_answer
              ex_answers.append(ex_extracted)
          else:
              ex_correct = False
              ex_answers.append(gold_answer - 1000)

          if fl_extracted is not None:
              fl_correct = fl_extracted == gold_answer
              fl_answers.append(fl_extracted)
          else:
              fl_correct = False
              fl_answers.append(gold_answer - 1000)

          print(f"[{i+1}/{len(dataset)}]({j+1}) EXACT_MATCH : gold={gold_answer} pred={ex_extracted} correct={ex_correct}")
          print(f"[{i+1}/{len(dataset)}]({j+1}) FLEX_EXTRACT: gold={gold_answer} pred={fl_extracted} correct={fl_correct}")
          print(f"[{i+1}/{len(dataset)}]({j+1}) thinking length={get_thinking_length(model_output)}")

          thinking_length += get_thinking_length(model_output)

          # most frequent answer
          ex_majority_vote[j] = max(set(ex_answers), key=ex_answers.count)
          fl_majority_vote[j] = max(set(fl_answers), key=fl_answers.count)

          # answer closest to true answer
          ex_best_of_m[j] = min(ex_answers, key=lambda x: abs(x - gold_answer))
          fl_best_of_m[j] = min(fl_answers, key=lambda x: abs(x - gold_answer))

          # determine majority vote and best-of-m picks for current token budget
          ex_maj_acc = sum([ex_majority_vote[k] == gold_answer for k in range(j+1)]) / (j+1)
          ex_bon_acc = sum([ex_best_of_m[k] == gold_answer for k in range(j+1)]) / (j+1)

          fl_maj_acc = sum([fl_majority_vote[k] == gold_answer for k in range(j+1)]) / (j+1)
          fl_bon_acc = sum([fl_best_of_m[k] == gold_answer for k in range(j+1)]) / (j+1)

          # only output results at 16000 or 32000 token budget
          if j == 3 or j == 7:
            print(f"m: {j+1} Exact Majority Acc: {ex_maj_acc} Exact Best-of-M Acc: {ex_bon_acc}")
            print(f"m: {j+1} Flex Majority Acc: {fl_maj_acc} Flex Best-of-M Acc: {fl_bon_acc}")
            print(f"Total Number of tokens used: {thinking_length}\n\n")

  return

In [ ]:
# Run parallel-scaling, with varying temperature
print("=" * 30)
print("Evaluation: Temperatures")
print("=" * 30)
run_temperature_eval(enable_thinking=True)

## Prompt Directive Variation

In [ ]:
def run_directive_eval(enable_thinking=True):
  model.to(device)
  model.eval()

  tokens_per_sample = 4000

  m_lengths = [1, 2, 4, 6, 8]

  break_directive = "Break this problem down into smaller parts: "
  diverse_directive = "Consider multiple approaches and choose the best one: "

  task_directives = [break_directive, diverse_directive]

  for i, example in enumerate(dataset):
      thinking_length = 0

      ex_majority_vote = {}
      ex_best_of_m = {}
      fl_majority_vote = {}
      fl_best_of_m = {}
      ex_answers = []
      fl_answers = []

      for directive in task_directives:
        print(f"Directive: {directive}, Progress: {i+1} out of {len(dataset)}")

        problem = example["prompt"]

        # Append prompt-directive to beginning of current problem
        directed_problem = f"{directive}\n\n{problem[0].get('content')}"
        gold_answer = int(example["label"])

        messages = [{"role": "system", "content": "You are a careful competition math assistant.  Always output your final answer in \\boxed{}."},
                    {"role": "user", "content": directed_problem}]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        for j in range(m_lengths[-1]):
          with torch.no_grad():
              output_ids = model.generate(
                  **inputs,
                  max_new_tokens=tokens_per_sample,
                  do_sample=True,
                  temperature=0.6,
                  top_p=0.95,
                  top_k=50,
              )

          # check if the generation has already finished (151645 is <|im_end|>)
          if 151645 not in output_ids[0][inputs.input_ids.size(-1):].tolist():
              # check if the thinking process has finished (151668 is </think>)
              # and prepare the second model input
              if 151668 not in output_ids[0][inputs.input_ids.size(-1):].tolist():
                  # print("thinking budget is reached")
                  early_stopping_text = "\n\nConsidering the limited time by the user, I have to stop thinking and give my final answer in \\boxed{}.\n</think>\n\n Final Answer: \\boxed{"
                  early_stopping_ids = tokenizer([early_stopping_text], return_tensors="pt", return_attention_mask=False).input_ids.to(model.device)
                  input_ids = torch.cat([output_ids, early_stopping_ids], dim=-1)

                  attention_mask = torch.ones_like(input_ids, dtype=torch.int64)

                  # second generation
                  output_ids = model.generate(
                      input_ids=input_ids,
                      attention_mask=attention_mask,
                      max_new_tokens=1,
                      do_sample=False,
                  )

          # Decode only the newly generated tokens
          generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
          model_output = tokenizer.decode(generated_ids, skip_special_tokens=True)

          ex_extracted = extract_answer(model_output, mode='exact_match')

          fl_extracted = extract_answer(model_output, mode='flexible_extract')

          if ex_extracted is not None:
              ex_correct = ex_extracted == gold_answer
              ex_answers.append(ex_extracted)
          else:
              ex_correct = False
              ex_answers.append(gold_answer - 1000)

          if fl_extracted is not None:
              fl_correct = fl_extracted == gold_answer
              fl_answers.append(fl_extracted)
          else:
              fl_correct = False
              fl_answers.append(gold_answer - 1000)

          print(f"[{i+1}/{len(dataset)}]({j+1}) EXACT_MATCH : gold={gold_answer} pred={ex_extracted} correct={ex_correct}")
          print(f"[{i+1}/{len(dataset)}]({j+1}) FLEX_EXTRACT: gold={gold_answer} pred={fl_extracted} correct={fl_correct}")
          print(f"[{i+1}/{len(dataset)}]({j+1}) thinking length={get_thinking_length(model_output)}")

          thinking_length += get_thinking_length(model_output)

          # most frequent answer
          ex_majority_vote[j] = max(set(ex_answers), key=ex_answers.count)
          fl_majority_vote[j] = max(set(fl_answers), key=fl_answers.count)

          # answer closest to true answer
          ex_best_of_m[j] = min(ex_answers, key=lambda x: abs(x - gold_answer))
          fl_best_of_m[j] = min(fl_answers, key=lambda x: abs(x - gold_answer))

          ex_maj_acc = sum([ex_majority_vote[k] == gold_answer for k in range(j+1)]) / (j+1)
          ex_bon_acc = sum([ex_best_of_m[k] == gold_answer for k in range(j+1)]) / (j+1)

          fl_maj_acc = sum([fl_majority_vote[k] == gold_answer for k in range(j+1)]) / (j+1)
          fl_bon_acc = sum([fl_best_of_m[k] == gold_answer for k in range(j+1)]) / (j+1)

          # only output results at 16000 or 32000 token budget
          if j == 3 or j == 7:
            print(f"m: {j+1} Exact Majority Acc: {ex_maj_acc} Exact Best-of-M Acc: {ex_bon_acc}")
            print(f"m: {j+1} Flex Majority Acc: {fl_maj_acc} Flex Best-of-M Acc: {fl_bon_acc}")
            print(f"Total Number of tokens used: {thinking_length}\n\n")

  return

In [ ]:
# Run parallel-scaling, with varying prompt directives
print("=" * 30)
print("Evaluation: Prompt Directives")
print("=" * 30)
run_directive_eval(enable_thinking=True)

# Sequential Scaling Strategies

Sequential Scaling Experiments 2.2 (Combined Stop and Wait Strategy)

In [9]:
# Setting thinking mode to true for all sequential experiments
ENABLE_THINKING = True
model.to(device)
model.eval()

# Getting token id for the </think> delimeter
eot_token_id = tokenizer.encode("</think>", add_special_tokens=False)[-1]

# Storing token budgets for performing experiments, creating dictionaries for storing metrics
thinking_token_budgets = [1024, 2048, 4096, 8192, 16384, 32000]
think_tokens_exact_acc = {} # accuracy of thinking tokens using exact answer mode
think_tokens_flexible_acc = {} # accuracy of thinking tokens using flexible extract mode
avg_total_tokens_to_exact_acc = {} # accuracy of average total tokens using exact answer mode
avg_total_tokens_to_flexible_acc = {} # accuracy of average total tokens using flexible extract mode

# Looping through each token budget, for each example in the dataset
for token_budget in thinking_token_budgets:

    # Storing metrics for token_budget
    records = []

    for i, example in enumerate(dataset):

        # Get problem and target answer
        problem = example["prompt"][0]["content"]
        gold_answer = int(example["label"])

        # Defining how the model should respond/behave based on "instructions"/input
        messages = [
            {"role": "system", "content": "You are a careful competition math assistant. Always output your final answer in \\boxed{}."},
            {"role": "user", "content": problem}
        ]

        # Convert messages to specific template for model to understand, then tokenize it
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        print(f"inputs: {tokenizer.decode(inputs['input_ids'][0])}")

        # Assigning variables to track thinking tokens used, and keep track of the input fed into the model and thinking text which the model
        # generates
        thinking_text = ""
        total_thinking_tokens = 0
        curr_input = inputs

        # Looping until we've exhausted the thinking token budget
        while total_thinking_tokens < token_budget:

            # Finding the number of remaining number thinking tokens needed to generate for this run
            # This is in case we generate the </think> token before we reach the token budget, in that case we still
            # have more tokens for the model to generate
            num_remain_tokens = token_budget - total_thinking_tokens

            # Generate the model output based on the current input being fed to the model. It will generate num_remain_tokens amount
            # of tokens, unless it generates an </think> tag, at which point it will stop generating. We will also employ the use of
            # KV Cache for faster generation. The current input will keep on getting updated as the model generates thinking tokens
            curr_output = model.generate(
                **curr_input,
                max_new_tokens=num_remain_tokens,
                do_sample=True,
                temperature=0.6,
                top_k=50,
                top_p=0.95,
                eos_token_id=eot_token_id,
                use_cache=True,
            )

            # Getting the newly generated token ids, decoding them back into english text, and checking if we've stopped early
            # (model generated </think> before token budget completes). If so, we'll remove the </think> tag as it's not part of our
            # total tokens
            newly_generated_ids = curr_output[0][curr_input["input_ids"].shape[1]:]
            new_text = tokenizer.decode(newly_generated_ids, skip_special_tokens=False)
            stop_early = eot_token_id in newly_generated_ids.tolist()

            # We will update the total new tokens generated accordingly. If we've generated the </think> token
            # and stopped, then we'll update the total new tokens, subtracted by 1 as we don't want to include that
            # token
            if stop_early:
              total_thinking_tokens += (len(newly_generated_ids) - 1)
              thinking_text += new_text.replace("</think>", "")
            else:
              total_thinking_tokens += len(newly_generated_ids)
              thinking_text += new_text

            print(f"new generated thinking text: {thinking_text}, total thinking token: {total_thinking_tokens}")

            # If the model generated </think> and hasn't reached the full token budget yet, then we'll perform the Wait strategy,
            # update the token count, and the current input which will get fed back into the model.
            # Otherwise, we will perform the Stop strategy, since we know at this point that the model has generated a token_budget
            # amount of thinking tokens (thinking tokens has reached the limit)
            if stop_early and total_thinking_tokens < token_budget:
                thinking_text += "Wait"
                total_thinking_tokens += 1
                curr_input = tokenizer(prompt + thinking_text, return_tensors="pt").to(device)
            else:
                break

        # We've finished generating the thinking text, so now we'll have the model generate the answer text
        answer_prompt = prompt + thinking_text + "</think>"
        print(f"answer prompt: {answer_prompt}")
        answer_inputs = tokenizer(answer_prompt, return_tensors="pt").to(device)

        # Generate the model output based on the the answer prompt (initial prompt + thinking text)
        answer_output = model.generate(
            **answer_inputs,
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.6,
            top_k=50,
            top_p=0.95,
            use_cache=True,
        )

        # Generating final model output by decoding answer tokens, and creating a the full model output using the prompt, thinking,
        # and answer text.
        answer_ids = answer_output[0][answer_inputs["input_ids"].shape[1]:]
        answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
        print(f"answer text: {answer_text}")
        model_output = thinking_text + "</think>" + answer_text
        print(f"model_output: {model_output}")

        # Calculating the number of thinking tokens used by the model
        total_tokens = total_thinking_tokens + len(answer_ids)

        # Extract both modes from same model output
        extracted_exact = extract_answer(model_output, mode="exact_match")
        extracted_flexible = extract_answer(model_output, mode="flexible_extract")
        
        # Checking whether the extracted answer matches the expected answer
        if extracted_exact is not None:
            correct_exact = (extracted_exact == gold_answer)
        else:
            correct_exact = False

        if extracted_flexible is not None:
            correct_flexible = (extracted_flexible == gold_answer)
        else:
            correct_flexible = False

        # Store metrics
        records.append({
            "problem": problem,
            "gold_answer": gold_answer,
            "model_output": model_output,
            "extracted_exact": extracted_exact,
            "extracted_flexible": extracted_flexible,
            "correct_exact": correct_exact,
            "correct_flexible": correct_flexible,
            "thinking_tokens": total_thinking_tokens,
            "total_tokens": total_tokens
        })

        print(f"[{i+1}/30] budget={token_budget} gold={gold_answer} pred_exact={extracted_exact} pred_flexible={extracted_flexible} correct_exact={correct_exact} correct_flexible={correct_flexible}")

    # Computing accuracies and average total tokens
    exact_acc = sum(r["correct_exact"] for r in records) / len(records)
    flexible_acc = sum(r["correct_flexible"] for r in records) / len(records)
    avg_total = sum(r["total_tokens"] for r in records) / len(records)

    # Store with token_budget as key
    think_tokens_exact_acc[token_budget] = exact_acc
    think_tokens_flexible_acc[token_budget] = flexible_acc
    avg_total_tokens_to_exact_acc[avg_total] = exact_acc
    avg_total_tokens_to_flexible_acc[avg_total] = flexible_acc

print(f"think_tokens_exact_acc: {think_tokens_exact_acc}")
print(f"avg_total_tokens_to_exact_acc: {avg_total_tokens_to_exact_acc}")
print(f"think_tokens_flexible_acc: {think_tokens_flexible_acc}")
print(f"avg_total_tokens_to_flexible_acc: {avg_total_tokens_to_flexible_acc}")



Streaming output truncated to the last 5000 lines.

Subtract (1) from (2):
$$
(x - \sqrt{41})^2 - x^2 = 9 \\
x^2 - 2x\sqrt{41} + 41 - x^2 = 9 \\
-2x\sqrt{41} + 41 = 9 \\
-2x\sqrt{41} = -32 \\
x = \frac{16}{\sqrt{41}} = \frac{16\sqrt{41}}{41}
$$

Now plug into (1):

$$
x^2 + y^2 = 80 \Rightarrow \left(\frac{16\sqrt{41}}{41}\right)^2 + y^2 = 80
$$

$$
\frac{256 \cdot 41}{1681} + y^2 = 80 \Rightarrow \frac{10496}{1681} + y^2 = 80
$$

$$
y^2 = 80 - \frac{10496}{1681} = \frac{134480 - 10496}{1681} = \frac{123984}{1681}
$$

$$
y = \sqrt{\frac{123984}{1681}} = \frac{\sqrt{123984}}{41}
$$

Now, we compute the area of triangle $ABC$ using vectors. Vectors:

- $\vec{AB} = (\sqrt{41}, 0, 0)$
- $\vec{AC} = \left(\frac{16\sqrt{41}}{41}, \frac{\sqrt{12398
[12/30] budget=1024 gold=104 pred_exact=None pred_flexible=41 correct_exact=False correct_flexible=False
inputs: <|im_start|>system
You are a careful competition math assistant. Always output your final answer in \boxed{}.<|im_end|>
<|im_start|>use